# Module 3: Extraction Agent & Ingestion

This notebook verifies the extraction of speaker roles, chunking logic, FinBERT sentiment tagging, embedding generation, and ingestion into the ChromaDB hot and cold stores.

In [ ]:
# Cell 1: Imports and Initialization
import sys
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

from transcript_fetcher import TranscriptFetcher
from extraction_agent import ExtractionAgent, EmbeddingService, SentimentTagger

# Pre-requisites initialization
embedding_service = EmbeddingService()
sentiment_tagger = SentimentTagger()
agent = ExtractionAgent(embedding_service, sentiment_tagger)

print("Extraction agent and embedding/sentiment services initialized successfully!")

In [ ]:
# Cell 2: Fetch and run extraction on Reliance Industries
fetcher = TranscriptFetcher()
reliance_result = await fetcher.fetch(
    company_name="Reliance Industries",
    bse_code="500325",
    nse_symbol="RELIANCE",
    quarter="Q4FY26"
)

extraction_summary = agent.run(reliance_result)
print("\n=== EXTRACTION SUMMARY ===")
for k, v in extraction_summary.items():
    print(f"{k}: {v}")

In [ ]:
# Cell 3: Fetch and run extraction on TCS (to show historical quarters in cold store)
# First fetch a mock historic quarter for TCS
tcs_q3 = await fetcher.fetch(
    company_name="TCS",
    bse_code="532540",
    nse_symbol="TCS",
    quarter="Q3FY26",
    force_fetch=True # Force to get a separate simulation/fetch
)

agent.run(tcs_q3)

# Now fetch next quarter Q4FY26 to trigger hot store promotion
tcs_q4 = await fetcher.fetch(
    company_name="TCS",
    bse_code="532540",
    nse_symbol="TCS",
    quarter="Q4FY26",
    force_fetch=True
)

agent.run(tcs_q4)
print("Ingestion and promotion completed for TCS.")

In [ ]:
# Cell 4: View Chunk Samples from Hot Store
print("\n--- Querying Hot Store ---")
hot_results = agent.hot_collection.get(
    where={"company": "Reliance Industries"},
    limit=2,
    include=["documents", "metadatas"]
)

for idx, (doc, meta) in enumerate(zip(hot_results["documents"], hot_results["metadatas"])):
    print(f"\nHot Chunk #{idx+1}:")
    print(f"Metadata: {meta}")
    print(f"Document Snippet: {doc[:300]}...")

In [ ]:
# Cell 5: View Chunk Samples from Cold Store
print("\n--- Querying Cold Store ---")
cold_results = agent.cold_collection.get(
    where={"company": "TCS"},
    limit=2,
    include=["documents", "metadatas"]
)

for idx, (doc, meta) in enumerate(zip(cold_results["documents"], cold_results["metadatas"])):
    print(f"\nCold Chunk #{idx+1}:")
    print(f"Metadata: {meta}")
    print(f"Document Snippet: {doc[:300]}...")